# Registro histórico: 05_efficientnet_baseline

Origem: `C67` — `Colab Notebooks/notebook_reorganizado.ipynb`.

EfficientNet-B0 com uma saída, BCEWithLogitsLoss e 10 épocas.

**Limitações conhecidas:** Registra avaliação e erro na previsão de amostras; média de acurácia por lote difere da agregação global.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


# EfficientNet Training - Notebook Reorganizado




# 1. Imports e setups

In [ ]:
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# Set random seeds for reproducibility
random.seed(91432)
np.random.seed(91432)
torch.manual_seed(91432)
if torch.cuda.is_available():
    torch.cuda.manual_seed(91432)

# 2. Importando o Dataset do Google Drive

In [ ]:
if not os.path.exists("/content/0-vgg"):
    print("Importando dataset...")
    from google.colab import drive
    drive.mount('/content/drive')

    # Copy dataset from Drive to Colab
    !cp -r "/content/drive/MyDrive/0-vgg" "/content/"
else:
    print("Dataset já importado")

# Setup paths
image_path = Path("/content/0-vgg/base_dir")
train_dir = image_path / "train_dir"
test_dir = image_path / "test_dir"
val_dir = image_path / "val_dir"

print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")

# 3. Download Helper Functions

In [ ]:
# Download helper functions from Learn PyTorch repo
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import accuracy_fn

# 4. Visualização e levantamento do dataset

In [ ]:
def count_images_per_class(directory):
    """Counts the number of image files per class in a directory."""
    class_counts = {}
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            image_count = 0
            for _, _, files in os.walk(class_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        image_count += 1
            class_counts[class_name] = image_count
    return class_counts

# Count images per class in each directory
train_class_counts = count_images_per_class(train_dir)
test_class_counts = count_images_per_class(test_dir)
val_class_counts = count_images_per_class(val_dir)

print("Number of images per class:")
print(f"  Train directory: {train_class_counts}")
print(f"  Test directory: {test_class_counts}")
print(f"  Validation directory: {val_class_counts}")

# Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))
print(f"\nTotal images found: {len(image_path_list)}")

# Display a random image
random_image_path = random.choice(image_path_list)
image_class = random_image_path.parent.stem
print(f"\nRandom image path: {random_image_path}")
print(f"Image class: {image_class}")

img = Image.open(random_image_path)
print(f"Image dimensions: {img.width}x{img.height}")
print(f"Image mode: {img.mode}")

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Sample Image - Class: {image_class}", fontsize=14)
plt.axis("off")
plt.show()

# 5. Data transforms para a arquitetura EfficientNet B0

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to slightly larger first
    transforms.RandomCrop(224),     # Random crop for data augmentation

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Test/validation transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data transforms created successfully!")

# 6. Visualização das imagens Transformadas

In [ ]:
def plot_transformed_images(image_paths, transform, n=3, seed=42):
    """Plot original and transformed images side by side."""
    if seed:
        random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)

    fig, axes = plt.subplots(n, 2, figsize=(10, 4*n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i, image_path in enumerate(random_image_paths):
        with Image.open(image_path) as f:
            # Original image
            axes[i, 0].imshow(f)
            axes[i, 0].set_title("Original")
            axes[i, 0].axis("off")

            # Transformed image
            transformed_image = transform(f).permute(1, 2, 0)
            # Denormalize for visualization
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            transformed_image = transformed_image * std + mean
            transformed_image = torch.clamp(transformed_image, 0, 1)

            axes[i, 1].imshow(transformed_image)
            axes[i, 1].set_title("Transformed")
            axes[i, 1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

    plt.tight_layout()
    plt.show()

plot_transformed_images(image_paths=image_path_list,
                       transform=train_transform,
                       n=2,
                       seed=42)

# 7. Criando os Dataloaders

In [ ]:
# Create datasets
train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)
val_data = datasets.ImageFolder(root=val_dir, transform=test_transform)

# Get class information
class_names = train_data.classes
class_dict = train_data.class_to_idx
print(f"Classes: {class_names}")
print(f"Class to index mapping: {class_dict}")
print(f"Dataset sizes - Train: {len(train_data)}, Test: {len(test_data)}, Val: {len(val_data)}")

# Optimized batch size for MobileNetV3
BATCH_SIZE = 32  # Smaller batch size for better convergence

# Create dataloaders
train_dataloader = DataLoader(dataset=train_data,
                             batch_size=BATCH_SIZE,
                             shuffle=True,
                             num_workers=2,  # For faster data loading
                             pin_memory=True if device == "cuda" else False)

val_dataloader = DataLoader(dataset=val_data,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            num_workers=2,
                            pin_memory=True if device == "cuda" else False)

test_dataloader = DataLoader(dataset=test_data,
                           batch_size=BATCH_SIZE,
                           shuffle=False,
                           num_workers=2,
                           pin_memory=True if device == "cuda" else False)

print(f"DataLoaders created - Train batches: {len(train_dataloader)}, Test batches: {len(test_dataloader)}, Val batches: {len(val_dataloader)}")

# 8. Criando o Modelo EfficientNet B0

In [ ]:
def create_efficientnet_b0_model(num_classes=1, pretrained=True):
    """Create and modify EfficientNet B0 for binary classification."""
    if pretrained:
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        print("Loaded EfficientNet_B0 with ImageNet pretrained weights")
    else:
        model = models.efficientnet_b0(weights=None)
        print("Created EfficientNet_B0 without pretrained weights")

    # (opcional) congelar camadas iniciais para transfer learning
    for param in model.features[:-3].parameters():
        param.requires_grad = False

    # Modificar o classifier para 1 saída (logit)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)

    print(f"Modified classifier for {num_classes} class")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    return model

# Create model
model = create_efficientnet_b0_model(num_classes=1, pretrained=True)
model = model.to(device)


# 9. Setup do treinamento

In [ ]:
# Loss function and optimizer optimized for MobileNetV3
loss_fn = nn.BCEWithLogitsLoss()

# Using AdamW optimizer with weight decay for better generalization
optimizer = optim.AdamW(model.parameters(),
                       lr=1e-4,  # Lower learning rate for fine-tuning
                       weight_decay=1e-4)  # Weight decay for regularization

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                               mode='min',
                                               factor=0.5,
                                               patience=3)

print("Training configuration setup complete!")
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")
print(f"Learning rate scheduler: {scheduler}")

# 10. Funções de treino e validação

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    model.train()
    train_loss, train_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for batch, (X, y) in enumerate(progress_bar):
        # Envia para device
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # y: [B] -> [B,1] e float pra BCE
        y_float = y.float().unsqueeze(1)

        # Forward: logits [B,1]
        logits = model(X)

        # Loss com logits
        loss = loss_fn(logits, y_float)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Probabilidades e predições binárias
        probs = torch.sigmoid(logits)          # [B,1]
        preds = (probs > 0.5).long()           # [B,1] 0/1

        # accuracy_fn espera [B] com ints
        acc = accuracy_fn(
            y_true=y,                          # já está [B] com 0/1
            y_pred=preds.squeeze(1)            # [B]
        )

        # Acumula métricas
        train_loss += loss.item()
        train_acc += acc

        progress_bar.set_postfix({
            'batch_loss': f'{loss.item():.4f}',
            'batch_acc': f'{acc:.2f}%'
        })

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc


def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    model.eval()
    val_loss, val_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Validation", leave=False)

    with torch.inference_mode():
        for batch, (X, y) in enumerate(progress_bar):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            y_float = y.float().unsqueeze(1)
            logits = model(X)
            loss = loss_fn(logits, y_float)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            acc = accuracy_fn(
                y_true=y,
                y_pred=preds.squeeze(1)
            )

            val_loss += loss.item()
            val_acc += acc

            progress_bar.set_postfix({
                'batch_loss': f'{loss.item():.4f}',
                'batch_acc': f'{acc:.2f}%'
            })

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    return val_loss, val_acc

# 11. Função de Treinamento Principal

In [ ]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          val_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
          epochs: int,
          device: torch.device = device):
    """Trains and validates a model for a given number of epochs."""
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "val_loss": [],
               "val_acc": []}

    best_val_loss = float('inf')
    best_val_acc = 0

    # Loop through training and validation steps for a number of epochs
    for epoch in tqdm(range(epochs), desc="Epochs"):
        train_loss, train_acc = train_step(model=model,
                                           data_loader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)

        val_loss, val_acc = val_step(model=model,
                                     data_loader=val_dataloader,
                                     loss_fn=loss_fn,
                                     device=device)

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)

        # Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"val_loss: {val_loss:.4f} | "
            f"val_acc: {val_acc:.4f}"
        )

        # Step the scheduler
        scheduler.step(val_loss)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            # torch.save(model.state_dict(), 'best_model.pth') # Uncomment to save model
            print(f"Saving best model with Val Acc: {best_val_acc:.4f}")

    # Return the filled results dictionary
    return results, best_val_acc

# 12. Treinamento do Modelo

In [ ]:
# Set the number of epochs
EPOCHS = 10

# Start the timer
start_time = timer()

# Start training
model_results, best_val_acc = train(model=model,
                                    train_dataloader=train_dataloader,
                                    val_dataloader=val_dataloader,
                                    optimizer=optimizer,
                                    loss_fn=loss_fn,
                                    scheduler=scheduler,
                                    epochs=EPOCHS,
                                    device=device)

# End the timer and calculate total training time
end_time = timer()
total_train_time = end_time - start_time

print(f"Total training time: {total_train_time:.2f} seconds")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

# 13. Resultados do Teste

In [ ]:
def test_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device = device):
    """Tests a model on a test dataset."""
    model.eval()
    test_loss, test_acc = 0, 0
    all_preds = []
    all_labels = []

    # Progress bar
    progress_bar = tqdm(data_loader, desc="Testing", leave=False)

    with torch.inference_mode():
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            y_logits = model(X)
            y_pred_prob = torch.sigmoid(y_logits).squeeze()

            # 2. Calculate loss
            loss = loss_fn(y_pred_prob, y.float())
            test_loss += loss.item()

            # Calculate accuracy
            y_pred_class = torch.round(y_pred_prob)
            test_acc += accuracy_fn(y_true=y.cpu(), y_pred=y_pred_class.cpu())

            # Store predictions and labels
            all_preds.extend(y_pred_class.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(data_loader)
    test_acc = test_acc / len(data_loader)

    return {"model_name": model.__class__.__name__,
            "model_loss": test_loss,
            "model_acc": test_acc,
            "predictions": all_preds,
            "labels": all_labels}

# Test the model
test_results = test_model(model=model,
                          data_loader=test_dataloader,
                          loss_fn=loss_fn,
                          device=device)

print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.4f}")

# 14. Salvando o Modelo e Histórico de Treinamento

In [ ]:
import json

# Define paths for saving
LOCAL_MODEL_PATH = Path("models")
DRIVE_MODEL_PATH = Path("/content/drive/MyDrive/Trained_Models")

LOCAL_MODEL_PATH.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist
DRIVE_MODEL_PATH.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

# Save model
MODEL_NAME = "EfficientNetB0_SkinLesion_Binary_Classifier_1.pth"

# --- Logic for local model saving ---
MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / MODEL_NAME

# Check if local model file already exists and append a number if necessary
counter = 1
original_model_name_local = MODEL_SAVE_PATH_LOCAL.stem
while MODEL_SAVE_PATH_LOCAL.exists():
    MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / f"{original_model_name_local}_{counter}{MODEL_SAVE_PATH_LOCAL.suffix}"
    counter += 1

print(f"Saving model locally to: {MODEL_SAVE_PATH_LOCAL}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_LOCAL)

# --- Logic for Google Drive model saving ---
MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / MODEL_NAME

# Check if Drive model file already exists and append a number if necessary
counter = 1
original_model_name_drive = MODEL_SAVE_PATH_DRIVE.stem
while MODEL_SAVE_PATH_DRIVE.exists():
    MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_model_name_drive}_{counter}{MODEL_SAVE_PATH_DRIVE.suffix}"
    counter += 1

print(f"Saving model to Google Drive: {MODEL_SAVE_PATH_DRIVE}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_DRIVE)

# Save training history
history = {
    "train_losses": model_results["train_loss"],
    "val_losses": model_results["val_loss"],
    "train_accs": model_results["train_acc"],
    "val_accs": model_results["val_acc"],
    "test_results": {
        "loss": test_results['model_loss'],
        "accuracy": test_results['model_acc']
    },
    "training_time": total_train_time,
    "epochs": EPOCHS,
    "best_val_acc": best_val_acc
}

# Define the path for the training history JSON in Google Drive
HISTORY_NAME = "training_history_EfficientNetB0_1.json"
HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / HISTORY_NAME

# Check if history file already exists and append a number if necessary
counter = 1
original_history_name = HISTORY_SAVE_PATH_DRIVE.stem
while HISTORY_SAVE_PATH_DRIVE.exists():
    HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_history_name}_{counter}{HISTORY_SAVE_PATH_DRIVE.suffix}"
    counter += 1

with open(HISTORY_SAVE_PATH_DRIVE, "w") as f:
    json.dump(history, f, indent=2)

print("Model and training history saved successfully to Google Drive!")

# 15. Fazendo Previsões

In [ ]:
def make_predictions(model: torch.nn.Module,
                    data: list,
                    device: torch.device = device):
    """Make predictions on a list of samples."""
    pred_probs = []
    model.eval()

    with torch.inference_mode():
        for sample in data:
            # Prepare sample
            sample = torch.unsqueeze(sample, dim=0).to(device)

            # Make prediction
            pred_logit = model(sample)
            pred_prob = torch.softmax(pred_logit.squeeze(), dim=0)
            pred_probs.append(pred_prob.cpu())

    return torch.stack(pred_probs)

def visualize_predictions(model, dataset, class_names, num_samples=6):
    """Visualize model predictions on random samples."""
    # Get random samples
    test_samples = []
    test_labels = []

    for sample, label in random.sample(list(dataset), k=num_samples):
        test_samples.append(sample)
        test_labels.append(label)

    # Make predictions
    pred_probs = make_predictions(model=model, data=test_samples)
    pred_classes = pred_probs.argmax(dim=1)

    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()

    for i, sample in enumerate(test_samples):
        # Denormalize image for display
        img = sample.permute(1, 2, 0)
        mean = torch.tensor([0.485, 0.456, 0.406])
        std = torch.tensor([0.229, 0.224, 0.225])
        img = img * std + mean
        img = torch.clamp(img, 0, 1)

        axes[i].imshow(img)

        pred_label = class_names[pred_classes[i]]
        truth_label = class_names[test_labels[i]]
        confidence = pred_probs[i].max().item() * 100

        title_text = f"Pred: {pred_label} ({confidence:.1f}%)\nTruth: {truth_label}"

        if pred_label == truth_label:
            axes[i].set_title(title_text, fontsize=12, color="green", fontweight='bold')
        else:
            axes[i].set_title(title_text, fontsize=12, color="red", fontweight='bold')

        axes[i].axis("off")

    plt.tight_layout()
    plt.suptitle("Model Predictions on Test Samples", fontsize=16, fontweight='bold', y=1.02)
    plt.show()

# Visualize predictions
print("Visualizing model predictions...")
visualize_predictions(model, val_data, class_names, num_samples=6)

# 16. Sumário e Resultados

In [ ]:
print("=" * 60)
print("📊 TRAINING SUMMARY")
print("=" * 60)
print(f"Model: EfficientNet_B0")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
print(f"Training Time: {total_train_time:.2f} seconds")
print(f"Epochs Trained: {EPOCHS}")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Final Test Accuracy: {test_results['model_acc']:.2f}%")
print("=" * 60)

# Performance metrics
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Generate classification report
y_true = test_results['labels']
y_pred = test_results['predictions']

print("\n📈 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()